# Fuga de clientes Telco: desarrollo en Google Colab

Este notebook reproduce en Colab el flujo de desarrollo del proyecto: revisión del dataset, entrenamiento con `train.py`, exportación del modelo a la app web, pruebas y una vista temporal de la app.

La app final está publicada en Vercel (https://radar-fuga-telco.vercel.app) y se puede desplegar con Docker en una VM de Google Cloud (ver `README.md`).

> El almacenamiento de Colab es temporal. Todo lo que importa vive en el repositorio de GitHub.

## 1. Clonar el repositorio e instalar dependencias

Instalamos las versiones exactas de `requirements.txt`, porque `model.joblib` depende de ellas. Es normal que pip muestre en rojo un aviso de conflicto con paquetes propios de Colab (`google-colab`, `numba`): el proyecto no los usa y la instalación igual termina bien.

Si más adelante una celda con `import pandas` falla con un error de versiones, use **Entorno de ejecución, Reiniciar sesión** y ejecute de nuevo desde la sección 2.

In [ ]:
!git clone https://github.com/melo11miguel/telco-churn-streamlit.git
%cd telco-churn-streamlit
!pip -q install -r requirements-dev.txt

## 2. Revisión mínima del dataset

Antes de modelar respondemos las preguntas de la guía:

- **¿Qué representa una fila?** Un cliente de una compañía de telecomunicaciones.
- **¿Cuál es el objetivo?** Predecir `Churn` (Yes/No): si el cliente canceló el servicio.
- **¿Qué variables pueden entrar al modelo?** Las que se conocen de un cliente activo: antigüedad, cargo mensual, contrato, servicios contratados y forma de pago.
- **¿Hay datos faltantes?** Solo en `TotalCharges` (11 vacíos), que además no usamos.
- **¿Hay numéricas y categóricas?** Sí: 3 numéricas y el resto categóricas.

In [ ]:
import pandas as pd

df = pd.read_csv("data/telco_churn.csv")
print(df.shape)
display(df.head())
display(df.dtypes)

In [ ]:
# TotalCharges viene como texto: los vacíos solo aparecen al convertirlo a número.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
display(df.isna().sum().sort_values(ascending=False).head(15))
display(df.describe(include="all").T)

In [ ]:
# Distribución del objetivo: las clases están desbalanceadas.
display(df["Churn"].value_counts())
display(df["Churn"].value_counts(normalize=True).round(3))

## 3. Evidencia para elegir variables

- `customerID` es un identificador: no entra.
- `TotalCharges` es casi antigüedad por cargo mensual, y sus vacíos son exactamente los clientes con `tenure = 0`.
- `gender` no cambia la tasa de fuga y es un atributo sensible.

In [ ]:
display(df[["tenure", "MonthlyCharges", "TotalCharges"]].corr().round(3))
print("tenure de los TotalCharges vacíos:", df.loc[df["TotalCharges"].isna(), "tenure"].unique())
display(pd.crosstab(df["gender"], df["Churn"], normalize="index").round(3))

In [ ]:
# Variables con más relación con la fuga
for col in ["Contract", "InternetService", "PaymentMethod", "TechSupport"]:
    display(pd.crosstab(df[col], df["Churn"], normalize="index").round(3))

## 4. Entrenar y evaluar con `train.py`

In [ ]:
!python train.py

In [ ]:
import json
print(json.dumps(json.load(open("metrics.json")), indent=2, ensure_ascii=False))

## 5. Exportar el modelo a la app web

La app web calcula la predicción en el navegador con los coeficientes del modelo, así que después de entrenar hay que exportarlos a `web/model.json`.

In [ ]:
!python export_web.py

## 6. Pruebas mínimas antes de Docker

- `pytest`: `web/model.json` coincide con `model.joblib`.
- `npm test`: la web da la misma probabilidad que scikit-learn en 300 casos y la interfaz responde bien a los clics.

In [ ]:
!python -m pytest -q
!npm install --silent && npm test

## 7. Ver la app web desde Colab

Servimos la carpeta `web/` y la abrimos con un túnel temporal. localtunnel pide una contraseña: es la IP pública de esta máquina de Colab, que imprime la primera línea.

La versión definitiva no depende de esto: está en https://radar-fuga-telco.vercel.app.

In [ ]:
!nohup python -m http.server 8000 --directory web > /content/web.log 2>&1 &
!sleep 2 && curl -s -o /dev/null -w "Servidor local: HTTP %{http_code}\n" http://localhost:8000/
!curl -s https://loca.lt/mytunnelpassword && echo
!npx --yes localtunnel --port 8000